# Creating AI without such libraries as Torch or TensorFlow

## Load dataset and STOI

In [ ]:
import numpy as np
import json

data = np.memmap("train.bin", dtype=np.uint32, mode="r")

with open("stoi.json", "r") as f:
    stoi = json.load(f)

print(f"Vocabulary length: {len(stoi)}")
print(f"Tokens length: {len(data)}")

## Split dataset into training and test

In [ ]:
data = data

n = int(0.9 * len(data))
train_data = data[:n]
test_data = data[n:]

In [ ]:
def get_batch(split, block_size, batch_size):
    source = train_data if split == "train" else test_data

    ix = np.random.randint(0, len(source) - block_size - 1, size=batch_size)

    x = np.array([source[i:i+block_size] for i in ix])
    y = np.array([source[i + 1 : i + block_size + 1] for i in ix])
    return x, y

# Streaming batching as there is too much data

In [1]:
import numpy as np
import random
import re
from tokenizers import Tokenizer


file_path = "train.txt"
tokenizer = Tokenizer.from_file("stories_tokenizer.json")

token_buffer = np.array([], dtype=np.int32)

CHUNK_SIZE = 500_000
MIN_BUFFER_TOKENS = 500_000
MAX_BUFFER_TOKENS = 700_000


def clean_text(text):
    text = text.replace("<|endoftext|>", " endoftext ")
    text = text.replace("<|end|>", " endoftext ")
    text = re.sub(r'<\|[^|]*\|>', '', text)
    text = ' '.join(text.split())
    return text


def load_random_chunk():
    with open(file_path, "rb") as f:
        f.seek(0, 2)
        file_size = f.tell()
        pos = random.randint(0, max(1, file_size - CHUNK_SIZE))
        f.seek(pos)
        chunk = f.read(CHUNK_SIZE)

    text = chunk.decode("utf-8", errors="ignore")
    text = clean_text(text)
    tokens = tokenizer.encode(text).ids
    return np.array(tokens, dtype=np.int32)


def refill_buffer():
    global token_buffer

    while len(token_buffer) < MIN_BUFFER_TOKENS:
        new_tokens = load_random_chunk()

        if len(new_tokens) < 100:
            continue

        token_buffer = np.concatenate([token_buffer, new_tokens]) if len(token_buffer) > 0 else new_tokens

        if len(token_buffer) > MAX_BUFFER_TOKENS:
            trim_start = random.randint(0, len(token_buffer) - MAX_BUFFER_TOKENS)
            token_buffer = token_buffer[trim_start:trim_start + MAX_BUFFER_TOKENS]


def get_batch(block_size, batch_size):
    global token_buffer

    refill_buffer()

    needed = block_size * batch_size + 1
    if len(token_buffer) < needed:
        raise ValueError(
            f"Buffer too small ({len(token_buffer)}) for a batch of "
            f"batch_size={batch_size}, block_size={block_size}."
        )

    max_start = len(token_buffer) - block_size - 1

    if random.random() < 0.7:
        starts = np.random.randint(0, max_start, size=batch_size)
    else:
        max_base = max_start - batch_size * block_size
        if max_base <= 0:
            starts = np.random.randint(0, max_start, size=batch_size)
        else:
            base = random.randint(0, max_base)
            starts = [base + i * block_size for i in range(batch_size)]

    x = np.stack([token_buffer[i:i + block_size] for i in starts])
    y = np.stack([token_buffer[i + 1:i + block_size + 1] for i in starts])

    consume_up_to = int(np.max(starts)) + block_size + 1
    token_buffer = token_buffer[consume_up_to:]

    return x, y

## Traing loop

In [2]:
import numpy as np
from tokenizers import Tokenizer

tokenizer = Tokenizer.from_file("stories_tokenizer.json")

def check_model_output(model, prompt, max_tokens):
    encoded_text = tokenizer.encode(prompt).ids
    context = np.array(encoded_text, dtype=np.uint32).reshape(1, -1)

    generated = model.generate(context, max_tokens, 1)

    output_text = tokenizer.decode(generated[0].tolist())

    print(output_text)

In [3]:
from NoTorchAI.GlobalState.Device import Device
from NoTorchAI.GlobalState.Quant import Quant
from NoTorchAI.Gradients.Adam import Adam
from NoTorchAI.LLM.MiniGPT import MiniGPT


d_model = 384
n_heads = 8
block_layers = 7
block_size = 128

batch_size = 56
vocabulary_size = 16_000

Device("gpu")
Quant(32)

gradient = Adam(
    lr=2.2e-4,
    warmup_steps=2500,
    min_lr=1e-5,
)

model = MiniGPT(vocab_size=vocabulary_size, 
                d_model=d_model, 
                block_size=block_size,
                n_layers=block_layers,
                n_heads=n_heads,
                gradient=gradient
            )

ema_loss = None

for step in range(500):
    xb, yb = get_batch(block_size=block_size, batch_size=batch_size)

    logits, loss = model.forward(xb, yb)

    gradient.t += 1
    model.backward()

    if ema_loss is None:
        ema_loss = loss
    else:
        ema_loss = 0.99 * ema_loss + 0.01 * loss

    if step == 1:
        print(f"step {step}, lr {gradient.get_lr():.6f}, loss {loss:.4f}, ema_loss {ema_loss:.4f}")

    if step % 100 == 0:
        check_model_output(model, "Tell me a story", 100)
        print(f"step {step}, lr {gradient.get_lr():.6f}, loss {loss:.4f}, ema_loss {ema_loss:.4f}")

    if step % 500 == 0:
        model.save("saved_model")

model.save("saved_model")

AttributeError: 'NormLayer' object has no attribute 'weights'

## Change model to instruct format

In [19]:
INSTRUCTIONS = [
    "Tell me a story about <|insert|>",
    "Can you tell me a story about <|insert|>?",
    "Write a short story about <|insert|>",
    "Make up a story about <|insert|>",
    "I want to hear a story about <|insert|>",
    "Create a fun story about <|insert|>",
    "Please tell me a story about <|insert|>",
    "Invent a story about <|insert|>",
    "Tell a bedtime story about <|insert|>",
    "Can you make a story about <|insert|>?",
    
    "Wanna hear a fun story? Tell me about <|insert|>",
    "Hey, tell me a story about <|insert|>",
    "Could you write a story about <|insert|>?",
    "Give me a story about <|insert|>",
    "Do you know a story about <|insert|>?",
    
    "Tell me a simple story about <|insert|>",
    "Tell me a children's story about <|insert|>",
    "Tell me a happy story about <|insert|>",
    "Tell me a funny story about <|insert|>",
    "Tell me an interesting story about <|insert|>",
    
    "Write a creative story about <|insert|>",
    "Write a nice story about <|insert|>",
    "Write a fun little story about <|insert|>",
    "Write a short and simple story about <|insert|>",
    
    "Imagine a story about <|insert|> and tell it to me",
    "Can you imagine a story about <|insert|>?",
    "Make up a creative story about <|insert|>",
    
    "Let’s hear a story about <|insert|>",
    "Tell me something about <|insert|> in story form",
    "Turn <|insert|> into a story",
    
    "Tell me a story involving <|insert|>",
    "Create a story where <|insert|> is important",
    "Write a story that includes <|insert|>",
    
    "Can you tell me a bedtime story about <|insert|>?",
    "Tell me a relaxing story about <|insert|>",
    
    "Tell me a story with <|insert|> in it",
    "Write a story where <|insert|> appears",
]

TOPICS = [
    "a dragon",
    "a little boy",
    "a magical forest",
    "a talking cat",
    "a brave knight",
    "a lost treasure",
    "a robot",
    "a lonely star",
]

In [109]:
import numpy as np
import random
import re
from tokenizers import Tokenizer


file_path = "train.txt"
tokenizer = Tokenizer.from_file("stories_tokenizer.json")

token_buffer = np.array([], dtype=np.int32)

CHUNK_SIZE = 500_000
MIN_BUFFER_TOKENS = 500_000
MAX_BUFFER_TOKENS = 700_000

def insert_instructions(text: str):
    stories = text.split(" endoftext")
    stories = [s.strip() for s in stories if len(s.strip()) > 200]

    for i in range(len(stories)):
        instruction = INSTRUCTIONS[random.randint(0, len(INSTRUCTIONS) - 1)]

        sentences = [s.strip() for s in stories[i].split(".") if len(s.strip()) > 20]

        if random.random() < 0.7 and len(sentences) > 0:
            sentence = random.choice(sentences)

            words = sentence.split()
            topic = " ".join(words[:5])

        else:
            topic = random.choice(TOPICS)
        
        instruction = instruction.replace("<|insert|>", topic)

        if random.random() < 0.3:
            stories[i] = "\nUser: " + instruction + "\nAssistant: " + stories[i]
        else:
            stories[i] = stories[i]
    
    return ' endoftext '.join(stories)


def clean_text(text):
    text = text.replace("<|endoftext|>", " endoftext ")
    text = text.replace("<|end|>", " endoftext ")
    text = insert_instructions(text)
    text = re.sub(r'<\|[^|]*\|>', '', text)
    text = ' '.join(text.split())
    return text


def load_random_chunk():
    with open(file_path, "rb") as f:
        f.seek(0, 2)
        file_size = f.tell()
        pos = random.randint(0, max(1, file_size - CHUNK_SIZE))
        f.seek(pos)
        chunk = f.read(CHUNK_SIZE)

    text = chunk.decode("utf-8", errors="ignore")
    text = clean_text(text)

    tokens = tokenizer.encode(text).ids
    return np.array(tokens, dtype=np.int32)


def refill_buffer():
    global token_buffer

    while len(token_buffer) < MIN_BUFFER_TOKENS:
        new_tokens = load_random_chunk()

        if len(new_tokens) < 100:
            continue

        token_buffer = np.concatenate([token_buffer, new_tokens]) if len(token_buffer) > 0 else new_tokens

        if len(token_buffer) > MAX_BUFFER_TOKENS:
            trim_start = random.randint(0, len(token_buffer) - MAX_BUFFER_TOKENS)
            token_buffer = token_buffer[trim_start:trim_start + MAX_BUFFER_TOKENS]


def get_batch(block_size, batch_size):
    global token_buffer

    refill_buffer()

    needed = block_size * batch_size + 1
    if len(token_buffer) < needed:
        raise ValueError(
            f"Buffer too small ({len(token_buffer)}) for a batch of "
            f"batch_size={batch_size}, block_size={block_size}."
        )

    max_start = len(token_buffer) - block_size - 1

    if random.random() < 0.7:
        starts = np.random.randint(0, max_start, size=batch_size)
    else:
        max_base = max_start - batch_size * block_size
        if max_base <= 0:
            starts = np.random.randint(0, max_start, size=batch_size)
        else:
            base = random.randint(0, max_base)
            starts = [base + i * block_size for i in range(batch_size)]

    x = np.stack([token_buffer[i:i + block_size] for i in starts])
    y = np.stack([token_buffer[i + 1:i + block_size + 1] for i in starts])

    consume_up_to = int(np.max(starts)) + block_size + 1
    token_buffer = token_buffer[consume_up_to:]

    return x, y

In [ ]:
ema_loss = None

gradient.lr = 1e-4
gradient.min_lr = 1e-5
gradient.warmup_steps = 2000
gradient.t = 0



for step in range(5_000, 15_000):

    xb, yb = get_batch(block_size=block_size, batch_size=batch_size)

    logits, loss = model.forward(xb, yb)

    gradient.t += 1
    model.backward()

    if ema_loss is None:
        ema_loss = loss
    else:
        ema_loss = 0.99 * ema_loss + 0.01 * loss

    if step < 5 or step % 200 == 0:
        print(f"step {step}, lr {gradient.get_lr():.6f}, loss {loss:.4f}, ema_loss {ema_loss:.4f}")

    if step % 500 == 0:
        check_model_output(model, "User: Tell me a story about a dragon\nAssistant:", 120)

    if step % 1000 == 0 and step > 0:
        model.save("finetuned_model")

model.save("finetuned_model")

step 0, lr 0.000010, loss 1.9124, ema_loss 1.9124
U ser : Tell me a story about a dragon A ss ist ant : Once upon a time , there was a little girl named Jane . Jane had a big , red ball . She loved to play with her ball every day . One day , Jane saw a small , lost cat . She was scared , but she wanted to help the cat . Jane had an idea . She took the cat home and gave it a soft cloth to clean the cat . The cat was very happy and thanked Jane . Jane and the cat became best friends . They played together every day . The cat was not frightened anymore . Jane was happy , and she learned that being brave can help you make friends
step 1, lr 0.000010, loss 1.8553, ema_loss 1.9119
step 2, lr 0.000010, loss 1.9590, ema_loss 1.9123
step 3, lr 0.000010, loss 1.9449, ema_loss 1.9127
step 4, lr 0.000010, loss 1.8903, ema_loss 1.9125
step 200, lr 0.000010, loss 1.9566, ema_loss 1.9210
step 400, lr 0.000020, loss 1.7822, ema_loss 1.8887
U ser : Tell me a story about a dragon A ss ist ant : One day 

CUDARuntimeError: cudaErrorUnknown: unknown error

In [ ]:
for step in range(5_000, 15_000):

    xb, yb = get_batch(block_size=block_size, batch_size=batch_size)

    logits, loss = model.forward(xb, yb)

    gradient.t += 1
    model.backward()

    if ema_loss is None:
        ema_loss = loss
    else:
        ema_loss = 0.99 * ema_loss + 0.01 * loss

    if step < 5 or step % 200 == 0:
        print(f"step {step}, lr {gradient.get_lr():.6f}, loss {loss:.4f}, ema_loss {ema_loss:.4f}")

    if step % 500 == 0:
        check_model_output(model, "User: Tell me a story about a dragon\nAssistant:", 120)

    if step % 1000 == 0 and step > 0:
        model.save("finetuned_model")

model.save("finetuned_model")

## Custom Decoder

In [ ]:
from NoTorchAI.LLM.MiniGPT import MiniGPT


# model = MiniGPT.__new__(MiniGPT)
# model: MiniGPT = model.load("saved_model")

# itos = {i: ch for ch, i in stoi.items()}

prompt = "History "
encoded_text = np.array([stoi[ch] for ch in prompt], dtype=np.uint32)
context = encoded_text.reshape(1, -1)

generated = generate(model, context, 30)

output_text = "".join([itos[int(num)] for num in generated[0]])
print(output_text)

## Hugging Face Decoder

In [1]:
from NoTorchAI.LLM.MiniGPT import MiniGPT
import numpy as np
from tokenizers import Tokenizer


tokenizer = Tokenizer.from_file("stories_tokenizer.json")

model = MiniGPT.__new__(MiniGPT)
model: MiniGPT = model.load("finetuned_model")


def check_model_output(model, prompt, max_tokens):
    encoded_text = tokenizer.encode(prompt).ids
    context = np.array(encoded_text, dtype=np.uint32).reshape(1, -1)

    generated = model.generate(context, max_tokens, 0.5)

    output_text = tokenizer.decode(generated[0].tolist())

    print(output_text)


check_model_output(model, "User: Tell me a story about Yarick being sick\nAssistant:", 200)

U ser : Tell me a story about Y ar ick being sick A ss ist ant : Once upon a time , there was a big , heavy rock . It lived in a small hole in the ground . Many people lived near the hole . One day , a little girl named Lily came to the hole . She wanted to see what was inside . She asked her mom if she could go in the hole . Her mom said yes , but she had to be careful . Lily was very careful with her heavy rock . She did not want to fall in the hole . She wanted to be safe now . So , she got her heavy rock and got her way out . Lily was happy that she could help her new friend . endoftext Once upon a time , in a small house , there lived a little girl named Lily . She had a big box of toys . She loved to play with them all day long . One day , Lily found a big box in her room . She was very happy and wanted to see what
